In [2]:
# ============================================================
# EXPERIMENT 2 — RESNET50 FEATURE EXTRACTION
# Brain Tumor MRI Classification
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from PIL import Image

from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split


# ============================================================
# 2. SETTINGS
# ============================================================

DATASET_DIR = Path("../dataset/brain_tumor_dataset")

BATCH_SIZE = 16
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)

class_to_label = {
    "glioma": 0,
    "healthy": 1,
    "meningioma": 2,
    "pituitary": 3
}

label_to_class = {
    0: "glioma",
    1: "healthy",
    2: "meningioma",
    3: "pituitary"
}


# ============================================================
# 3. CHECK DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if device.type == "cpu":
    print("Running ResNet50 on CPU.")
    print("This can take some time.")


# ============================================================
# 4. CREATE IMAGE DATAFRAME
# ============================================================

records = []

for class_name, label in class_to_label.items():

    class_folder = DATASET_DIR / class_name

    for image_path in class_folder.glob("*.jpg"):

        records.append({
            "path": str(image_path),
            "class": class_name,
            "label": label
        })

df = pd.DataFrame(records)

print("\nOriginal images:", len(df))


# ============================================================
# 5. REMOVE EXACT DUPLICATES
# ============================================================

def file_hash(path):

    hasher = hashlib.md5()

    with open(path, "rb") as f:

        while chunk := f.read(8192):
            hasher.update(chunk)

    return hasher.hexdigest()


hashes = {}

for path in df["path"]:

    h = file_hash(path)

    hashes.setdefault(h, []).append(path)


duplicate_paths = set()
duplicate_groups = 0

for paths in hashes.values():

    if len(paths) > 1:

        duplicate_groups += 1

        # Keep first copy
        # Exclude remaining copies
        duplicate_paths.update(paths[1:])


clean_df = df[
    ~df["path"].isin(duplicate_paths)
].copy()

clean_df.reset_index(drop=True, inplace=True)


print("Duplicate groups:", duplicate_groups)
print("Duplicate copies removed:", len(duplicate_paths))
print("Clean images:", len(clean_df))


# ============================================================
# 6. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

train_df, temp_df = train_test_split(
    clean_df,
    test_size=0.30,
    stratify=clean_df["label"],
    random_state=RANDOM_SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=RANDOM_SEED
)


print("\nDATASET SPLIT")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))


# ============================================================
# 7. RESNET50 PREPROCESSING
# ============================================================

# Load the official ImageNet-pretrained ResNet50 weights.
weights = ResNet50_Weights.DEFAULT

# Use the preprocessing associated with those weights.
resnet_transform = weights.transforms()


# ============================================================
# 8. PYTORCH DATASET
# ============================================================

class BrainTumorDataset(Dataset):

    def __init__(self, dataframe, transform):

        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform


    def __len__(self):

        return len(self.dataframe)


    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(
            row["path"]
        ).convert("RGB")

        image = self.transform(image)

        label = row["label"]

        return image, label


# ============================================================
# 9. CREATE DATASETS
# ============================================================

train_dataset = BrainTumorDataset(
    train_df,
    resnet_transform
)

val_dataset = BrainTumorDataset(
    val_df,
    resnet_transform
)

test_dataset = BrainTumorDataset(
    test_df,
    resnet_transform
)


# ============================================================
# 10. CREATE DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ============================================================
# 11. LOAD PRETRAINED RESNET50
# ============================================================

print("\nLoading pretrained ResNet50...")

resnet = resnet50(
    weights=weights
)


# ============================================================
# 12. REMOVE FINAL CLASSIFIER
# ============================================================

# Original ResNet50:
#
# Image
#   ↓
# ResNet layers
#   ↓
# 2048 features
#   ↓
# Fully connected layer
#   ↓
# 1000 ImageNet classes
#
# We don't want the 1000 ImageNet classes.
#
# We want the 2048 learned features.

resnet.fc = nn.Identity()

resnet = resnet.to(device)

resnet.eval()


print("ResNet50 loaded.")
print("Final classifier removed.")
print("Feature size: 2048")


# ============================================================
# 13. FEATURE EXTRACTION FUNCTION
# ============================================================

def extract_features(model, loader):

    all_features = []
    all_labels = []

    with torch.no_grad():

        for batch_number, (images, labels) in enumerate(loader):

            images = images.to(device)

            features = model(images)

            all_features.append(
                features.cpu().numpy()
            )

            all_labels.append(
                labels.numpy()
            )

            if (batch_number + 1) % 20 == 0:

                print(
                    "Processed batches:",
                    batch_number + 1
                )


    features = np.concatenate(
        all_features,
        axis=0
    )

    labels = np.concatenate(
        all_labels,
        axis=0
    )

    return features, labels


# ============================================================
# 14. EXTRACT TRAINING FEATURES
# ============================================================

print("\n================================")
print("EXTRACTING TRAINING FEATURES")
print("================================")

X_train, y_train = extract_features(
    resnet,
    train_loader
)

print(
    "Training feature shape:",
    X_train.shape
)


# ============================================================
# 15. EXTRACT VALIDATION FEATURES
# ============================================================

print("\n================================")
print("EXTRACTING VALIDATION FEATURES")
print("================================")

X_val, y_val = extract_features(
    resnet,
    val_loader
)

print(
    "Validation feature shape:",
    X_val.shape
)


# ============================================================
# 16. EXTRACT TEST FEATURES
# ============================================================

print("\n================================")
print("EXTRACTING TEST FEATURES")
print("================================")

X_test, y_test = extract_features(
    resnet,
    test_loader
)

print(
    "Testing feature shape:",
    X_test.shape
)


# ============================================================
# 17. SAVE FEATURES
# ============================================================

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

np.savez_compressed(
    RESULTS_DIR / "resnet50_features.npz",
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test
)


# ============================================================
# 18. FINAL CHECK
# ============================================================

print("\n================================")
print("RESNET50 FEATURE EXTRACTION DONE")
print("================================")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nSaved to:")
print(RESULTS_DIR / "resnet50_features.npz")

Device: cpu
Running ResNet50 on CPU.
This can take some time.

Original images: 7023
Duplicate groups: 300
Duplicate copies removed: 426
Clean images: 6597

DATASET SPLIT
Training: 4617
Validation: 990
Testing: 990

Loading pretrained ResNet50...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\Asus/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [01:23<00:00, 1.23MB/s]


ResNet50 loaded.
Final classifier removed.
Feature size: 2048

EXTRACTING TRAINING FEATURES
Processed batches: 20
Processed batches: 40
Processed batches: 60
Processed batches: 80
Processed batches: 100
Processed batches: 120
Processed batches: 140
Processed batches: 160
Processed batches: 180
Processed batches: 200
Processed batches: 220
Processed batches: 240
Processed batches: 260
Processed batches: 280
Training feature shape: (4617, 2048)

EXTRACTING VALIDATION FEATURES
Processed batches: 20
Processed batches: 40
Processed batches: 60
Validation feature shape: (990, 2048)

EXTRACTING TEST FEATURES
Processed batches: 20
Processed batches: 40
Processed batches: 60
Testing feature shape: (990, 2048)

RESNET50 FEATURE EXTRACTION DONE
X_train: (4617, 2048)
y_train: (4617,)
X_val: (990, 2048)
y_val: (990,)
X_test: (990, 2048)
y_test: (990,)

Saved to:
..\results\resnet50_features.npz
